# ЛР-6. Random Forest

Датасет: CareerCon 2019 (Kaggle) — IMU-сенсоры робота, 9 типов поверхности.  
Деревья: **не-бинарные** (категориальные → N-way split) и **не-сбалансированные**.  
Требуется обосновать ограничение глубины.

In [1]:
import numpy as np
import pandas as pd

from mlcore.utils.notebook import lab_paths, setup_plotting
from mlcore.preprocessing.encoding import label_encode
from mlcore.preprocessing.splitting import train_test_split
from mlcore.trees.criteria import entropy, information_gain
from mlcore.metrics import accuracy
from mlcore.metrics.report import print_classification_metrics, save_confusion_matrix, save_multiclass_roc
from mlcore.utils.timing import timer

ROOT, _, ASSETS = lab_paths(6)
setup_plotting()
import matplotlib.pyplot as plt
DATA = ROOT / 'labs/lr-6/data'

## 1. Загрузка данных

Данные необходимо скачать с Kaggle:  
https://www.kaggle.com/competitions/career-con-2019/data  
Файлы: `X_train.csv`, `y_train.csv`, `X_test.csv` → `labs/lr-6/data/`

In [2]:
x_train_path = DATA / 'X_train.csv'
y_train_path = DATA / 'y_train.csv'

assert x_train_path.exists(), f'Download X_train.csv to {DATA}'
assert y_train_path.exists(), f'Download y_train.csv to {DATA}'

X_raw = pd.read_csv(x_train_path)
y_raw = pd.read_csv(y_train_path)
print(f'X_raw: {X_raw.shape}, y_raw: {y_raw.shape}')
print(f'Columns X: {X_raw.columns.tolist()}')
print(f'Columns y: {y_raw.columns.tolist()}')
print(f'\nSurface classes ({y_raw["surface"].nunique()}):\n{y_raw["surface"].value_counts()}')

X_raw: (487680, 13), y_raw: (3810, 3)
Columns X: ['row_id', 'series_id', 'measurement_number', 'orientation_X', 'orientation_Y', 'orientation_Z', 'orientation_W', 'angular_velocity_X', 'angular_velocity_Y', 'angular_velocity_Z', 'linear_acceleration_X', 'linear_acceleration_Y', 'linear_acceleration_Z']
Columns y: ['series_id', 'group_id', 'surface']

Surface classes (9):
surface
concrete                  779
soft_pvc                  732
wood                      607
tiled                     514
fine_concrete             363
hard_tiles_large_space    308
soft_tiles                297
carpet                    189
hard_tiles                 21
Name: count, dtype: int64


## 2. Feature Engineering

Агрегируем 128 временных шагов в статистики по каждому каналу:  
mean, std, min, max, range, median, q25, q75 × 10 каналов = 80 числовых + group_id (категориальный) = 81.

In [3]:
sensor_cols = [c for c in X_raw.columns if c not in ['row_id', 'series_id', 'measurement_number']]
print(f'Sensor columns ({len(sensor_cols)}): {sensor_cols}')

agg_funcs = {
    'mean': 'mean', 'std': 'std', 'min': 'min', 'max': 'max',
    'range': lambda x: x.max() - x.min(),
    'median': 'median', 'q25': lambda x: x.quantile(0.25), 'q75': lambda x: x.quantile(0.75),
}

grouped = X_raw.groupby('series_id')[sensor_cols]
parts = []
for stat_name, func in agg_funcs.items():
    part = grouped.agg(func)
    part.columns = [f'{c}_{stat_name}' for c in sensor_cols]
    parts.append(part)

X_feat = pd.concat(parts, axis=1)
X_feat = X_feat.loc[y_raw['series_id']]  # align

# Add group_id as categorical
group_ids = y_raw.set_index('series_id')['group_id']
X_feat['group_id'] = group_ids.values

print(f'Feature matrix: {X_feat.shape}')

Sensor columns (10): ['orientation_X', 'orientation_Y', 'orientation_Z', 'orientation_W', 'angular_velocity_X', 'angular_velocity_Y', 'angular_velocity_Z', 'linear_acceleration_X', 'linear_acceleration_Y', 'linear_acceleration_Z']
Feature matrix: (3810, 81)


In [4]:
# Encode target
y_encoded, y_map = label_encode(y_raw['surface'].values)
print(f'Target classes: {y_map}')

# Identify categorical feature index (group_id = last column)
cat_indices = {X_feat.shape[1] - 1}  # group_id

X_np = X_feat.values.astype(object)  # object to allow mixed types
# Convert numeric columns to float
for j in range(X_np.shape[1]):
    if j not in cat_indices:
        X_np[:, j] = X_np[:, j].astype(np.float64)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_np, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
# Validation set from training data for hyperparameter tuning (depth selection)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr
)
print(f'Train: {X_tr.shape}, Val: {X_val.shape}, Test: {X_te.shape}')

Target classes: {'carpet': 0, 'concrete': 1, 'fine_concrete': 2, 'hard_tiles': 3, 'hard_tiles_large_space': 4, 'soft_pvc': 5, 'soft_tiles': 6, 'tiled': 7, 'wood': 8}
Train: (2439, 81), Val: (609, 81), Test: (762, 81)


## 3. Decision Tree (не-бинарные, не-сбалансированные)

- Категориальные признаки (group_id) → **N-way split** (один потомок на каждое значение)
- Числовые признаки → бинарный split по порогу
- Дерево **не-сбалансировано**: разные ветви имеют разную глубину

In [5]:
class TreeNode:
    __slots__ = ('feature', 'threshold', 'categories', 'children', 'is_leaf',
                 'prediction', 'class_proba', 'n_classes')
    def __init__(self, n_classes):
        self.feature = None
        self.threshold = None
        self.categories = None
        self.children = {}  # value -> TreeNode (multi-way) or 'left'/'right' -> TreeNode
        self.is_leaf = False
        self.prediction = 0
        self.class_proba = np.zeros(n_classes)
        self.n_classes = n_classes


class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=5, max_features='sqrt',
                 categorical_indices=None, random_state=42, n_classes=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.cat_idx = categorical_indices or set()
        self.rng = np.random.default_rng(random_state)
        self.n_classes = n_classes or 0
        self.root = None

    def fit(self, X, y):
        if self.n_classes == 0:
            self.n_classes = len(np.unique(y))
        self.root = self._build(X, y, 0)
        return self

    def _make_leaf(self, y):
        node = TreeNode(self.n_classes)
        node.is_leaf = True
        node.prediction = np.bincount(y, minlength=self.n_classes).argmax()
        node.class_proba = np.bincount(y, minlength=self.n_classes).astype(float)
        node.class_proba /= node.class_proba.sum() + 1e-12
        return node

    def _build(self, X, y, depth):
        if (len(y) < self.min_samples_split or len(np.unique(y)) == 1 or
                (self.max_depth is not None and depth >= self.max_depth)):
            return self._make_leaf(y)

        n_feat = X.shape[1]
        if self.max_features == 'sqrt':
            k = max(1, int(np.sqrt(n_feat)))
        else:
            k = n_feat
        candidates = self.rng.choice(n_feat, size=min(k, n_feat), replace=False)

        best_gain, best_feat, best_split = 0, None, None

        for j in candidates:
            if j in self.cat_idx:
                # Multi-way split
                vals = np.unique(X[:, j])
                if len(vals) < 2:
                    continue
                groups = {v: y[X[:, j] == v] for v in vals}
                children_labels = list(groups.values())
                ig = information_gain(y, children_labels, criterion='entropy')
                if ig > best_gain:
                    best_gain, best_feat = ig, j
                    best_split = {'type': 'cat', 'groups': {v: X[:, j] == v for v in vals}}
            else:
                col = X[:, j].astype(np.float64)
                unique_vals = np.unique(col)
                if len(unique_vals) < 2:
                    continue
                # Subsample thresholds
                if len(unique_vals) > 50:
                    thresholds = np.quantile(col, np.linspace(0.02, 0.98, 50))
                else:
                    thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2
                for t in thresholds:
                    left = col <= t
                    if left.sum() == 0 or (~left).sum() == 0:
                        continue
                    ig = information_gain(y, [y[left], y[~left]], criterion='entropy')
                    if ig > best_gain:
                        best_gain, best_feat = ig, j
                        best_split = {'type': 'num', 'threshold': t}

        if best_gain == 0 or best_feat is None:
            return self._make_leaf(y)

        node = TreeNode(self.n_classes)
        node.feature = best_feat
        node.prediction = np.bincount(y, minlength=self.n_classes).argmax()
        node.class_proba = np.bincount(y, minlength=self.n_classes).astype(float)
        node.class_proba /= node.class_proba.sum() + 1e-12

        if best_split['type'] == 'cat':
            node.categories = list(best_split['groups'].keys())
            for val, mask in best_split['groups'].items():
                node.children[val] = self._build(X[mask], y[mask], depth + 1)
        else:
            node.threshold = best_split['threshold']
            col = X[:, best_feat].astype(np.float64)
            left = col <= node.threshold
            node.children['left'] = self._build(X[left], y[left], depth + 1)
            node.children['right'] = self._build(X[~left], y[~left], depth + 1)

        return node

    def _predict_one(self, x, node):
        if node.is_leaf:
            return node.class_proba
        if node.categories is not None:  # categorical
            val = x[node.feature]
            if val in node.children:
                return self._predict_one(x, node.children[val])
            return node.class_proba  # fallback for unseen
        else:  # numeric
            if float(x[node.feature]) <= node.threshold:
                return self._predict_one(x, node.children['left'])
            return self._predict_one(x, node.children['right'])

    def predict_proba(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

print('DecisionTree defined.')

DecisionTree defined.


In [6]:
class RandomForest:
    def __init__(self, n_estimators=100, max_depth=10, min_samples_split=5,
                 max_features='sqrt', categorical_indices=None, random_state=42, n_classes=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.cat_idx = categorical_indices or set()
        self.rng = np.random.default_rng(random_state)
        self.n_classes = n_classes or 0
        self.trees: list[DecisionTree] = []

    def fit(self, X, y):
        n = len(y)
        if self.n_classes == 0:
            self.n_classes = len(np.unique(y))
        self.trees = []
        for i in range(self.n_estimators):
            boot_idx = self.rng.choice(n, size=n, replace=True)
            tree = DecisionTree(
                max_depth=self.max_depth, min_samples_split=self.min_samples_split,
                max_features=self.max_features, categorical_indices=self.cat_idx,
                random_state=self.rng.integers(1_000_000), n_classes=self.n_classes
            )
            tree.fit(X[boot_idx], y[boot_idx])
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        all_proba = np.array([tree.predict_proba(X) for tree in self.trees])
        return all_proba.mean(axis=0)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

print('RandomForest defined.')

RandomForest defined.


## 4. Обоснование глубины

Эксперимент: обучаем лес при разных max_depth, сравниваем train и validation accuracy.  
Validation set выделен из обучающей выборки (80/20), тестовая выборка не используется при подборе гиперпараметров.

In [7]:
depths = [3, 5, 7, 10, 15, 20, None]
train_acc, val_acc = [], []

for d in depths:
    rf = RandomForest(n_estimators=30, max_depth=d, categorical_indices=cat_indices,
                      n_classes=len(y_map), random_state=42)
    rf.fit(X_tr, y_tr)
    ta = accuracy(y_tr, rf.predict(X_tr))
    va = accuracy(y_val, rf.predict(X_val))
    train_acc.append(ta)
    val_acc.append(va)
    d_str = str(d) if d is not None else 'None'
    print(f'depth={d_str:>4s}: train_acc={ta:.3f}, val_acc={va:.3f}')

fig, ax = plt.subplots(figsize=(8, 5))
x_labels = [str(d) if d is not None else '∞' for d in depths]
ax.plot(x_labels, train_acc, 'o-', label='Train')
ax.plot(x_labels, val_acc, 's-', label='Validation')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Depth Justification')
ax.legend()
fig.tight_layout()
fig.savefig(ASSETS / 'depth_justification.png', dpi=150)
plt.close(fig)
print('Saved depth_justification.png')

depth=   3: train_acc=0.948, val_acc=0.936
depth=   5: train_acc=0.987, val_acc=0.962
depth=   7: train_acc=1.000, val_acc=0.987
depth=  10: train_acc=1.000, val_acc=0.993
depth=  15: train_acc=1.000, val_acc=0.993
depth=  20: train_acc=1.000, val_acc=0.992
depth=None: train_acc=1.000, val_acc=0.992
Saved depth_justification.png


In [8]:
# Best depth by validation accuracy
best_depth_idx = np.argmax(val_acc)
best_depth = depths[best_depth_idx]
print(f'Selected depth: {best_depth} (val_acc={val_acc[best_depth_idx]:.3f})')

Selected depth: 10 (val_acc=0.993)


## 5. Финальная модель

In [9]:
final_rf = RandomForest(n_estimators=100, max_depth=best_depth,
                        categorical_indices=cat_indices,
                        n_classes=len(y_map), random_state=42)
with timer('Random Forest training'):
    final_rf.fit(X_tr, y_tr)

y_pred = final_rf.predict(X_te)
y_proba = final_rf.predict_proba(X_te)

[timer] (Random Forest training): 114.8160s


In [10]:
print_classification_metrics(y_te, y_pred)

Accuracy: 0.9987
Precision: 0.9992
Recall: 0.9989
F1-score: 0.9991


{'Accuracy': 0.9986876640419947,
 'Precision': 0.999244142101285,
 'Recall': 0.9989212513484359,
 'F1-score': 0.9990787757706927}

In [11]:
inv_map = {v: k for k, v in y_map.items()}
class_names = [inv_map[i] for i in range(len(y_map))]

save_confusion_matrix(y_te, y_pred, class_names, ASSETS / 'confusion_matrix.png')
print('Saved confusion_matrix.png')

Saved confusion_matrix.png


In [12]:
auc_dict = save_multiclass_roc(y_te, y_proba, class_names, ASSETS / 'roc_multiclass.png')

## 6. Выводы

1. Реализован Random Forest с не-бинарными деревьями: категориальный признак `group_id` разбивается на N потомков (по одному на каждое уникальное значение), числовые признаки — бинарно по порогу. Это позволяет учитывать природу данных: категориальные значения не имеют порядка, поэтому бинарный split (≤ / >) на них семантически некорректен.
2. Деревья не-сбалансированы — глубина ветвей зависит от данных, разные поддеревья растут до разной глубины.
3. Глубина обоснована экспериментально: при малых значениях (3–5) модель недообучена, при неограниченной глубине train accuracy выходит на 1.0, но validation accuracy не растёт — признак переобучения. Оптимум выбран по максимуму val accuracy на отдельном валидационном множестве (не на тесте).
4. Feature engineering (агрегация 128 временных шагов в 8 статистик × 10 каналов) критически важен для качества — без него модель работала бы с сырыми временными рядами разной длины.
5. Random Forest устойчивее одиночного дерева за счёт bagging (bootstrap-выборки) и feature subsampling (√n признаков на каждый split), что снижает дисперсию предсказаний.
6. Метрики (precision, recall, F1) вычислены с macro-усреднением: каждый из 9 классов вносит равный вклад, что важно при неравных размерах классов.